## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.m

In [1]:
import os
from dotenv import load_dotenv

groq_api_key=os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq
model=ChatGroq(model='llama-3.1-8b-instant', api_key=groq_api_key, temperature=0.1)
model

c:\Users\Dell\Desktop\codes\Udemy_GenAI\LangChain\lang\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000026D60AA1000>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000026D60AA16F0>, model_name='llama-3.1-8b-instant', temperature=0.1, model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content='Hi I am Dhyan Sher and I am a chief AI Engineer')])

AIMessage(content="Nice to meet you, Dhyan Sher. As a chief AI Engineer, I'm sure you have a deep understanding of the latest advancements in artificial intelligence and its applications. What specific areas of AI are you currently working on or interested in?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 48, 'total_tokens': 97, 'completion_time': 0.070150356, 'completion_tokens_details': None, 'prompt_time': 0.004269063, 'prompt_tokens_details': None, 'queue_time': 0.049064762, 'total_time': 0.074419419}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ffe44-911d-7181-b655-76ba9baf5402-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 49, 'total_tokens': 97})

In [6]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Dhyan and I am a Chief AI Engineer"),
        AIMessage(content="Hello Dhyan! It's nice to meet you. \n\nAs a Chief AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content="Your name is Dhyan, and you're a Chief AI Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 119, 'total_tokens': 134, 'completion_time': 0.015569299, 'completion_tokens_details': None, 'prompt_time': 0.007413219, 'prompt_tokens_details': None, 'queue_time': 0.048674721, 'total_time': 0.022982518}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ffe46-5833-7793-a727-ca75dbc9f16f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 119, 'output_tokens': 15, 'total_tokens': 134})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [9]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory(session_id=session_id)
    return store[session_id]

with_history = RunnableWithMessageHistory(model, get_session_history=get_session_history)

c:\Users\Dell\Desktop\codes\Udemy_GenAI\LangChain\lang\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [10]:
config={'configurable':{'session_id':'chat1'}}

In [12]:
response=with_history.invoke([
    HumanMessage(content="Hi , My name is Dhyan and I am a Chief AI Engineer"),
    AIMessage(content="Hello Dhyan! It's nice to meet you. \n\nAs a Chief AI Engineer, what kind of projects are you working on these days? \n\nI'm always eager to learn more about the exciting work being done in the field of AI.\n"),
    HumanMessage(content="Hey What's my name and what do I do?")
], config=config)

In [13]:
response.content

"Your name is Dhyan, and you're a Chief AI Engineer."

In [16]:
with_history.invoke([
    HumanMessage(content='Whats my name?')],config=config
)

AIMessage(content='Your name is Dhyan.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 255, 'total_tokens': 262, 'completion_time': 0.007907635, 'completion_tokens_details': None, 'prompt_time': 0.017466801, 'prompt_tokens_details': None, 'queue_time': 0.047358128, 'total_time': 0.025374436}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ffe4d-550e-7291-a667-428af01acc6d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 255, 'output_tokens': 7, 'total_tokens': 262})

In [17]:
## change the session id to chat2 and check the history is not there
config={'configurable':{'session_id':'chat2'}}

In [18]:
response=with_history.invoke([
    HumanMessage(content="Hi whats my name?")], config=config)

In [19]:
response.content

"I don't have any information about your name. I'm a large language model, I don't have the ability to retain information about individual users or their personal details. Each time you interact with me, it's a new conversation and I don't have any prior knowledge about you. If you'd like to share your name with me, I'd be happy to chat with you!"